# AOI 瑕疵分類 — 優化版完整 Pipeline（訓練 + 驗證 + 推論）
由 Topic 18 Notebook A / B 整合優化：
1. 預訓練權重（Transfer Learning）   2. 分層切出驗證集（Stratified Split）
3. 訓練/評估使用一致的 Resize          4. AdamW + OneCycleLR + Label Smoothing
5. AMP 混合精度                        6. 保存最佳模型（依驗證準確率）
7. num_workers / pin_memory 加速讀檔   8. 推論 TTA（水平翻轉）
9. 混淆矩陣 + Overkill / Underkill 分析 10. 產生 AIdea 上傳檔

In [ ]:
# Step 0: 下載資料（Colab）
!pip install -q --upgrade gdown
!gdown https://drive.google.com/uc?id=1tovCO2gsjesjJ8OsfHgahyt-buY34dk0
!unzip -q aoi-dataset.zip && rm aoi-dataset.zip

In [ ]:
import os
import time
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
@dataclass
class CFG:
    data_dir: str = "."
    model_name: str = "efficientnet_b0"
    pretrained: bool = True
    img_size: int = 384          # 原圖 512x512，224 會損失細小瑕疵
    num_classes: int = 6
    batch_size: int = 32
    epochs: int = 15
    lr: float = 1e-3
    weight_decay: float = 1e-4
    label_smoothing: float = 0.1
    val_ratio: float = 0.2
    num_workers: int = 2
    use_amp: bool = True
    tta: bool = True
    seed: int = 42
    ckpt: str = "aoi_best.pth"
    submission: str = "submission.csv"


cfg = CFG()
CLASS_NAMES = ["normal", "void", "horizontal", "vertical", "edge", "particle"]


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

In [ ]:
# Dataset：在 __init__ 先把欄位轉成 list，避免每次 __getitem__ 呼叫 df.iloc
class AOIDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_dir: str, transform=None):
        self.ids = df["ID"].tolist()
        self.labels = df["Label"].astype(int).tolist()
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        path = os.path.join(self.img_dir, self.ids[idx])
        image = Image.open(path).convert("RGB")   # 灰階 → 3 通道，配合預訓練模型
        if self.transform is not None:
            image = self.transform(image)
        return image, self.labels[idx]

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# 注意：不可做 90 度旋轉——會把「水平瑕疵」變成「垂直瑕疵」，標籤就錯了
train_tf = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),   # 與訓練相同尺寸
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

In [ ]:
# 分層切分：每個類別在 train / val 中比例相同
df_all = pd.read_csv(os.path.join(cfg.data_dir, "train.csv"))
df_tr, df_va = train_test_split(
    df_all, test_size=cfg.val_ratio, stratify=df_all["Label"], random_state=cfg.seed)

img_dir = os.path.join(cfg.data_dir, "train_images")
loader_kw = dict(batch_size=cfg.batch_size, num_workers=cfg.num_workers,
                 pin_memory=(device.type == "cuda"),
                 persistent_workers=cfg.num_workers > 0)
train_loader = DataLoader(AOIDataset(df_tr, img_dir, train_tf), shuffle=True,
                          drop_last=True, **loader_kw)
val_loader = DataLoader(AOIDataset(df_va, img_dir, eval_tf), shuffle=False, **loader_kw)
print(f"train={len(df_tr)}  val={len(df_va)}  steps/epoch={len(train_loader)}")

In [ ]:
def build_model(name: str, num_classes: int, pretrained: bool = True) -> nn.Module:
    weights = "DEFAULT" if pretrained else None
    model = getattr(models, name)(weights=weights)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)   # 只換最後一層
    return model


model = build_model(cfg.model_name, cfg.num_classes, cfg.pretrained).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"{cfg.model_name}: {n_params / 1e6:.2f} M parameters")

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=cfg.lr, epochs=cfg.epochs, steps_per_epoch=len(train_loader))
use_amp = cfg.use_amp and device.type == "cuda"
scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()                      # OneCycle：每個 batch 更新一次
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        n += labels.size(0)
    return total_loss / n, correct / n


@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            logits = model(images)
            total_loss += criterion(logits, labels).item() * labels.size(0)
        all_preds.append(logits.argmax(1).cpu())
        all_labels.append(labels.cpu())
    preds, labels = torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()
    return total_loss / len(labels), (preds == labels).mean(), preds, labels

In [ ]:
best_acc = 0.0
for epoch in range(1, cfg.epochs + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader)
    va_loss, va_acc, _, _ = evaluate(model, val_loader)
    flag = ""
    if va_acc > best_acc:
        best_acc = va_acc
        torch.save(model.state_dict(), cfg.ckpt)
        flag = "  ★ saved"
    print(f"[{epoch:02d}/{cfg.epochs}] train {tr_loss:.4f}/{tr_acc:.4f} | "
          f"val {va_loss:.4f}/{va_acc:.4f} | {time.time() - t0:.0f}s{flag}")
print(f"best val acc = {best_acc:.4f}")

In [ ]:
# 驗證集分析：混淆矩陣 + Overkill / Underkill
model.load_state_dict(torch.load(cfg.ckpt, map_location=device))
_, va_acc, va_preds, va_labels = evaluate(model, val_loader)
print(confusion_matrix(va_labels, va_preds))
print(classification_report(va_labels, va_preds, target_names=CLASS_NAMES, digits=4))

overkill = int(((va_labels == 0) & (va_preds != 0)).sum())    # 良品被判為瑕疵
underkill = int(((va_labels != 0) & (va_preds == 0)).sum())   # 瑕疵被判為良品（漏檢）
print(f"overkill={overkill}  underkill={underkill}")

In [ ]:
# 推論：test set + TTA（原圖與水平翻轉的機率平均）
@torch.inference_mode()
def predict(model, loader, tta=True):
    model.eval()
    outputs = []
    for images, _ in loader:
        images = images.to(device)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            prob = model(images).softmax(1)
            if tta:
                prob = (prob + model(torch.flip(images, dims=[3])).softmax(1)) / 2
        outputs.append(prob.float().cpu())
    return torch.cat(outputs).argmax(1).numpy()


df_test = pd.read_csv(os.path.join(cfg.data_dir, "test.csv"))
test_loader = DataLoader(AOIDataset(df_test, os.path.join(cfg.data_dir, "test_images"), eval_tf),
                         shuffle=False, **loader_kw)
test_preds = predict(model, test_loader, tta=cfg.tta)

df_out = df_test[["ID"]].copy()
df_out["Label"] = test_preds
df_out.to_csv(cfg.submission, index=False)
print(df_out["Label"].value_counts().sort_index())
print("saved:", cfg.submission)